# NB04 — PGLS validation

**Goal:** Aggregate MAG density to genus level and run PGLS to check consistency with the P1 result (comprehensive_metal_ecology).

**H5 (PGLS consistency):** The per-Mb density × local metal mobility association at genus level is directionally consistent with P1 (β < 0 for niche breadth, or positive for metal availability).

**Steps:**
1. Aggregate `ko_per_mb_primary` to genus by median.
2. Merge with metal mobility (median PF1_Cu per genus sampling location).
3. Join to P1 PGLS feature matrix (genus-level `levins_b_z`, phylogenetic tree).
4. Run PGLS: `levins_b_z ~ ko_per_mb_z + log_PF1_Cu_z` via `pgls_utils.run_pgls`.
5. Compare β(ko_per_mb_z) to P1 value (−0.021).

**Output:** `data/pgls_validation_results.csv`, summary printed to notebook.

In [1]:
print("NB04 executing — PGLS validation against P1.")

NB04 executing — PGLS validation against P1.


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

# pgls_utils from comprehensive_metal_ecology
_REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(_REPO_ROOT / 'comprehensive_metal_ecology' / 'scripts'))
from pgls_utils import run_pgls, pgls_results_table, fdr_correct

DATA_DIR = Path.cwd().parent / 'data'
CME_DATA = _REPO_ROOT / 'comprehensive_metal_ecology' / 'data'

In [3]:
# MAG feature matrix
df = pd.read_parquet(DATA_DIR / 'mag_feature_matrix.parquet')

# P1 genus-level data: 01_pgls_input_bacteria.csv has genus_lower + mean_levins_B_std
feature_matrix = pd.read_csv(CME_DATA / '01_pgls_input_bacteria.csv')
feature_matrix = feature_matrix.rename(columns={
    'genus_lower': 'genus',
    'mean_levins_B_std': 'levins_b_z',   # already z-scored in P1
})

print(f"MAG feature matrix: {len(df):,} MAGs")
print(f"P1 feature matrix: {len(feature_matrix):,} genera")
print(f"P1 columns: {list(feature_matrix.columns)}")

MAG feature matrix: 15,957 MAGs
P1 feature matrix: 1,574 genera
P1 columns: ['genus', 'ko_per_mb_primary', 'mean_genome_mb', 'levins_b_z', 'phylum', 'kingdom', 'predictor_z', 'genome_mb_z']


In [4]:
# The MAG feature matrix needs a genus-level taxonomic label.
# Assume a 'genus' column is available from genome_metadata (joined in NB01).
# If not available, GTDB-Tk taxonomic assignment is required first.

if 'genus' not in df.columns:
    raise ValueError(
        "'genus' column not found in mag_feature_matrix.parquet. "
        "Join GTDB-Tk taxonomy before running NB04."
    )

genus_df = df.groupby('genus').agg(
    ko_per_mb_primary=('ko_per_mb_primary', 'median'),
    log_PF1_Cu=('PF1_Cu', lambda x: np.log1p(x.median())),
    n_mags=('mag_id', 'count'),
).reset_index()

print(f"Genera with ≥1 MAG: {len(genus_df)}")
print(genus_df[['genus', 'ko_per_mb_primary', 'log_PF1_Cu', 'n_mags']].head())

Genera with ≥1 MAG: 1934
             genus  ko_per_mb_primary  log_PF1_Cu  n_mags
0  0-14-0-80-60-11          31.434190    0.067207      37
1   01-FULL-38-12b         197.060255    0.060264       2
2   01-FULL-45-10b          79.733847    0.058538       1
3   01-FULL-45-15b         169.336574    0.058538       1
4   01-FULL-45-34b         182.623702    0.058538       1


In [5]:
# P1 genus names are lowercase; GTDB/Spark genus names are mixed-case — align before merge
genus_df['genus'] = genus_df['genus'].str.lower()

# Keep only the columns we need from P1 to avoid collision with genus_df's ko_per_mb_primary
p1_slim = feature_matrix[['genus', 'levins_b_z']].copy()

merged = p1_slim.merge(genus_df, on='genus', how='inner')
print(f"Genera in both P1 and MAG dataset: {len(merged)}")

# Z-score MAG-derived columns (nan_policy='omit' preserves length for NaN rows)
# levins_b_z is already z-scored from P1 — no recomputation needed
for col in ['ko_per_mb_primary', 'log_PF1_Cu']:
    merged[col + '_z'] = stats.zscore(merged[col], nan_policy='omit')

Genera in both P1 and MAG dataset: 294


In [6]:
TREE_PATH = CME_DATA / 'gtdb_bac_genus_pruned.tree'

pgls_out = run_pgls(
    df=merged,
    tree_path=TREE_PATH,
    response='levins_b_z',
    predictors=['ko_per_mb_primary_z', 'log_PF1_Cu_z'],
    taxon_col='genus',
)

results_df = pgls_results_table([pgls_out])
results_df['p_value_fdr'] = fdr_correct(results_df['p_value'].dropna())
results_df.to_csv(DATA_DIR / 'pgls_validation_results.csv', index=False)

print(results_df.to_string(index=False))

ko_beta = pgls_out['betas']['ko_per_mb_primary_z']
P1_BETA = -0.021
consistent = (np.sign(ko_beta) == np.sign(P1_BETA))
print(f"\nH5 directional consistency (β={ko_beta:.4f} vs P1 β={P1_BETA}): "
      f"{'CONSISTENT' if consistent else 'INCONSISTENT'}")

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


label   response           predictor   n  lambda_est      beta       SE    t_stat  p_value       r2  delta_aic_vs_null  p_value_fdr
      levins_b_z ko_per_mb_primary_z 254      0.8077 -0.010651 0.008588 -1.240202 0.216059 0.158345              -2.39     0.216059
      levins_b_z        log_PF1_Cu_z 254      0.8077 -0.013303 0.006939 -1.917056 0.056366 0.158345              -2.39     0.112732

H5 directional consistency (β=-0.0107 vs P1 β=-0.021): CONSISTENT
